In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
import sys
# Add parent directory to path to import modules from one level up
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, parent_dir)
from sklearn.datasets import fetch_openml

def load_fico_heloc_openml(data_id=45554, make_minority_one=True):
    """
    Loads the OpenML 'FICO-HELOC-cleaned' dataset (commonly referenced as OpenML d/45554),
    returns a DataFrame with columns: ENROLID, target, <features...>.

    - Handles RiskPerformance (Good/Bad) if present.
    - Treats -8 and -9 as missing (per common HELOC handling).
    """
    bunch = fetch_openml(data_id=data_id, as_frame=True)  # or name="FICO-HELOC-cleaned"
    X = bunch.data.copy()
    y = bunch.target.copy()

    # Unify into a single frame
    df = X.copy()
    df["__y__"] = y

    # Convert label to 0/1
    # Common: RiskPerformance in {"Good","Bad"} (Bad = worse outcome)
    y_str = df["__y__"].astype(str).str.strip().str.lower()
    if set(y_str.unique()).issubset({"good", "bad"}):
        df["target"] = (y_str == "bad").astype(int)
    else:
        # fallback: if already numeric or other coding
        y_num = pd.to_numeric(df["__y__"], errors="coerce")
        if set(y_num.dropna().unique()).issubset({0, 1}):
            df["target"] = y_num.astype(int)
        else:
            # map minority value to 1
            vc = y_num.value_counts(dropna=True)
            minority_val = vc.idxmin()
            df["target"] = (y_num == minority_val).astype(int)

    df = df.drop(columns=["__y__"])

    # Treat -8 / -9 as missing (common in HELOC usage)
    # (works whether the data is int/float or object)
    df = df.replace({-8: np.nan, -9: np.nan, "-8": np.nan, "-9": np.nan})

    # Add ENROLID
    df["ENROLID"] = np.arange(1, len(df) + 1, dtype=np.int64)
    cols = ["ENROLID", "target"] + [c for c in df.columns if c not in ["ENROLID", "target"]]
    df = df[cols]

    # Ensure numeric features where possible
    feature_cols = [c for c in df.columns if c not in ["ENROLID", "target"]]
    for c in feature_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Optionally ensure minority class is labeled 1 (your code assumes cases=1)
    if make_minority_one:
        vc = df["target"].value_counts()
        if vc.get(1, 0) > vc.get(0, 0):
            df["target"] = 1 - df["target"]

    print("\n=== FICO HELOC LOADED (OpenML) ===")
    print("Shape:", df.shape)
    print("Target counts:\n", df["target"].value_counts())
    return df

def setup_feature_columns(df, target_col="target"):
    """
    APS: typically all features are numeric after coercion.
    """
    feature_cols = [c for c in df.columns if c not in ["ENROLID", target_col]]

    # After coercion, treat as numeric
    cat_cols = df[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()
    if len(cat_cols) > 0:
        # In APS, this usually indicates you missed numeric coercion.
        print("⚠️ Warning: found non-numeric feature cols (expected none):", cat_cols[:10])

    numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

    # Binary detection (rare in APS but keep consistent with your framework)
    bin_cols = []
    for col in numeric_cols:
        u = df[col].dropna().unique()
        if len(u) <= 2 and set(u).issubset({0, 1, 0.0, 1.0}):
            bin_cols.append(col)

    true_num_cols = [c for c in numeric_cols if c not in bin_cols]

    print(f"\n=== APS COLUMN SETUP ===")
    print(f"Total features: {len(feature_cols)}")
    print(f"Categorical columns: {len(cat_cols)}")
    print(f"Binary columns: {len(bin_cols)}")
    print(f"True numeric columns: {len(true_num_cols)}")

    return feature_cols, cat_cols, true_num_cols, bin_cols

def create_train_test_split(df, target_col='target', test_size=0.3, val_size=0.5, random_state=123):
    """
    Create train/val/test splits similar to the main script.
    Returns: train_df, val_df, test_df
    """
    # First split: train vs (val+test)
    train_df, temp_df = train_test_split(
        df, 
        test_size=test_size, 
        stratify=df[target_col], 
        random_state=random_state
    )
    
    # Second split: val vs test (from temp_df)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=val_size,
        stratify=temp_df[target_col],
        random_state=random_state
    )
    
    print(f"\n=== DATA SPLITS ===")
    print(f"Train: {len(train_df):,} samples")
    print(f"  - Minority: {(train_df[target_col] == 1).sum():,}")
    print(f"  - Majority: {(train_df[target_col] == 0).sum():,}")
    print(f"Val: {len(val_df):,} samples")
    print(f"  - Minority: {(val_df[target_col] == 1).sum():,}")
    print(f"  - Majority: {(val_df[target_col] == 0).sum():,}")
    print(f"Test: {len(test_df):,} samples")
    print(f"  - Minority: {(test_df[target_col] == 1).sum():,}")
    print(f"  - Majority: {(test_df[target_col] == 0).sum():,}")
    
    return train_df, val_df, test_df


def precompute_case_control_distances(
    train_df, target_col, feature_cols,
    cat_columns, true_num_columns,
    dataset_name, seed=123
):
    import os
    import numpy as np
    from precompute_distances import compute_distances_batched, save_distances_hdf5

    cases = train_df[train_df[target_col] == 1].copy()
    controls = train_df[train_df[target_col] == 0].copy()

    X_cases = cases[feature_cols].copy()
    X_controls = controls[feature_cols].copy()

    print(f"Cases (minority): {len(cases):,}")
    print(f"Controls (majority): {len(controls):,}")

    used_cat = [c for c in cat_columns if c in X_controls.columns]
    used_num = [c for c in true_num_columns if c in X_controls.columns]

    # Ensure we don't drop any features unintentionally
    covered = set(used_cat + used_num)
    dropped = [c for c in feature_cols if c not in covered]
    assert len(dropped) == 0, f"These features would be dropped by the preprocessor: {dropped[:10]}"

    preprocessor = get_preprocessor_with_impute(
        categorical_cols=used_cat,
        numeric_cols=used_num,
        verbose=True
    )

    X_controls_processed = preprocessor.fit_transform(X_controls).astype(np.float32)
    X_cases_processed = preprocessor.transform(X_cases).astype(np.float32)

    assert np.isfinite(X_controls_processed).all()
    assert np.isfinite(X_cases_processed).all()

    distances = compute_distances_batched(
        X_controls_processed, X_cases_processed,
        batch_size=1000, dtype=np.float32
    )

    print(f"  Distance matrix shape: {distances.shape}")
    print(f"  Distance range: [{distances.min():.3f}, {distances.max():.3f}]")
    print(f"  Distance mean: {distances.mean():.3f}")

    output_dir = "./precomputed_distances"
    os.makedirs(output_dir, exist_ok=True)
    h5_path = os.path.join(output_dir, f"distances_{dataset_name}_seed_{seed}.h5")

    majority_enrolids = controls["ENROLID"].to_numpy()
    minority_enrolids = cases["ENROLID"].to_numpy()

    save_distances_hdf5(distances, majority_enrolids, minority_enrolids, h5_path, compression="gzip")
    print(f"\n✓ Saved distances to: {h5_path}")

    return h5_path, X_controls_processed, majority_enrolids

def make_biased_observed_dataset(
    df_full: pd.DataFrame,
    target_col: str = "target",
    pos_label: int = 1,
    target_pos_frac: float = 0.03,   # e.g., 0.02–0.05
    seed: int = 123,
    keep_all_negatives: bool = True,
):
    """
    Create a biased 'observed' dataset by randomly dropping positives so that
    P(Y=1) ~ target_pos_frac, mimicking observation bias toward negatives.

    Assumes df_full already contains ENROLID and target.
    """
    assert 0 < target_pos_frac < 0.5, "target_pos_frac should be in (0, 0.5) for 'rare positives'."
    rng = np.random.default_rng(seed)

    df = df_full.copy()
    pos = df[df[target_col] == pos_label]
    neg = df[df[target_col] != pos_label]

    n_neg = len(neg)
    # Want: n_pos_keep / (n_pos_keep + n_neg_keep) = target_pos_frac
    # If keeping all negatives: n_neg_keep = n_neg
    n_neg_keep = n_neg if keep_all_negatives else int(np.floor(len(df) * (1 - target_pos_frac)))
    if not keep_all_negatives:
        # Optional: downsample negatives too (not needed for your story)
        neg_keep_idx = rng.choice(neg.index.to_numpy(), size=n_neg_keep, replace=False)
        neg = df.loc[neg_keep_idx]
        n_neg = len(neg)

    n_pos_keep = int(np.floor((target_pos_frac / (1 - target_pos_frac)) * n_neg))
    n_pos_keep = max(1, min(n_pos_keep, len(pos)))

    pos_keep_idx = rng.choice(pos.index.to_numpy(), size=n_pos_keep, replace=False)
    pos_kept = df.loc[pos_keep_idx]

    df_obs = pd.concat([neg, pos_kept], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)

    frac = df_obs[target_col].mean()
    print("\n=== BIASED OBSERVED HELOC DATASET ===")
    print(f"Requested target positive fraction: {target_pos_frac:.3f}")
    print(f"Achieved positive fraction:        {frac:.3f}")
    print("Counts:\n", df_obs[target_col].value_counts().sort_index())
    return df_obs

df_heloc = load_fico_heloc_openml()

In [18]:
# ============================================================
# HELOC (biased observed) — undersampling + OCT finetune + eval
# ============================================================
import os
import numpy as np
import pandas as pd
import importlib

import model_IAI
importlib.reload(model_IAI)
from model_IAI import finetune_oct_impute, evaluate_binary_oct

import kcenter_hyperparameter_search_global
importlib.reload(kcenter_hyperparameter_search_global)
from kcenter_hyperparameter_search_global import run_global_kcenter_matching, build_undersampled_dataset

from sklearn.impute import SimpleImputer
from imblearn.under_sampling import RandomUnderSampler  # add others if needed

RESULTS_DIR = "./heloc_obs005"
os.makedirs(RESULTS_DIR, exist_ok=True)

TRAIN_TEST_SEED = 123
TARGET_POS_FRAC = 0.05
DATASET_NAME = f"fico_heloc_obs{int(TARGET_POS_FRAC*1000):03d}"

# 1) Bias then split
df_heloc_obs = make_biased_observed_dataset(df_heloc, target_col="target",
                                            target_pos_frac=TARGET_POS_FRAC,
                                            seed=TRAIN_TEST_SEED)

feature_cols_heloc, CAT_COLUMNS_HELOC, TRUE_NUM_COLUMNS_HELOC, BIN_COLUMNS_HELOC = setup_feature_columns(df_heloc_obs)
train_heloc, val_heloc, test_heloc = create_train_test_split(df_heloc_obs, random_state=TRAIN_TEST_SEED)

X_val, y_val = val_heloc[feature_cols_heloc], val_heloc["target"]
X_test, y_test = test_heloc[feature_cols_heloc], test_heloc["target"]

# 2) Distances on TRAIN
h5_path_heloc, _, _ = precompute_case_control_distances(
    train_heloc, "target", feature_cols_heloc,
    CAT_COLUMNS_HELOC, TRUE_NUM_COLUMNS_HELOC,
    dataset_name=DATASET_NAME, seed=TRAIN_TEST_SEED
)

# 3) Baseline OCT (raw train)
OCT_DEPTHS = [5, 7, 9]
OCT_MINBUCKETS = [25, 50, 100]
OCT_CPS = [1e-5, 1e-4, 1e-3, 1e-2]

baseline_model, baseline_params, _, baseline_preproc, baseline_featnames = finetune_oct_impute(
    X_train=train_heloc[feature_cols_heloc],
    y_train=train_heloc["target"],
    X_val=X_val, y_val=y_val,
    categorical_cols=CAT_COLUMNS_HELOC,
    numeric_cols=TRUE_NUM_COLUMNS_HELOC,
    depths=OCT_DEPTHS, minbuckets=OCT_MINBUCKETS, cps=OCT_CPS,
)
baseline_metrics = evaluate_binary_oct(
    baseline_model, X_test, y_test,
    baseline_preproc, baseline_featnames, X_val_df = X_val, y_val=y_val,
    results_dir=RESULTS_DIR, save_suffix="baseline"
)

# 4) Your method (single config as you set)
SEED_METHODS = ["random"]
CASE_WEIGHTINGS = [None]
USE_ADAPTIVE_POOL = [True]
MATCHING_RATIOS = [1]

dnn_dir = f"./precomputed_distances/global_dnn_{DATASET_NAME}_seed_{TRAIN_TEST_SEED}"
os.makedirs(dnn_dir, exist_ok=True)

all_rows = []

for seed_method in SEED_METHODS:
    for case_weighting in CASE_WEIGHTINGS:
        for use_adaptive_pool in USE_ADAPTIVE_POOL:
            for matching_ratio in MATCHING_RATIOS:

                matching_result = run_global_kcenter_matching(
                    train_pd=train_heloc,
                    target_col="target",
                    feature_cols=feature_cols_heloc,
                    pn_h5_path=h5_path_heloc,
                    matching_ratio=matching_ratio,
                    case_weighting=case_weighting,
                    use_adaptive_pool=use_adaptive_pool,
                    seed_method=seed_method,
                    CAT_COLUMNS=CAT_COLUMNS_HELOC,
                    TRUE_NUM_COLUMNS=TRUE_NUM_COLUMNS_HELOC,
                    COST_COLUMNS=None,
                    dnn_out_dir=dnn_dir,
                )

                undersampled_train = build_undersampled_dataset(
                    train_pd=train_heloc,
                    matching_result=matching_result,
                    target_col="target",
                    matching_ratio=matching_ratio,
                )

                model_c, params_c, _, preproc_c, featnames_c = finetune_oct_impute(
                    X_train=undersampled_train[feature_cols_heloc],
                    y_train=undersampled_train["target"],
                    X_val=X_val, y_val=y_val,
                    categorical_cols=CAT_COLUMNS_HELOC,
                    numeric_cols=TRUE_NUM_COLUMNS_HELOC,
                    depths=OCT_DEPTHS, minbuckets=OCT_MINBUCKETS, cps=OCT_CPS,
                )

                val_metrics = evaluate_binary_oct(model_c, X_val, y_val, preproc_c, featnames_c, X_val_df = X_val, y_val=y_val,
    results_dir=RESULTS_DIR, save_suffix="validation")

                all_rows.append({
                    "seed_method": seed_method,
                    "case_weighting": case_weighting,
                    "use_adaptive_pool": use_adaptive_pool,
                    "matching_ratio": matching_ratio,
                    "val_pr_auc": float(val_metrics.get("pr_auc", np.nan)),
                    "_model": model_c, "_preproc": preproc_c, "_featnames": featnames_c
                })

# best on val
best_idx = int(np.nanargmax([r["val_pr_auc"] for r in all_rows]))
best = all_rows[best_idx]

test_metrics = evaluate_binary_oct(best["_model"], X_test, y_test, best["_preproc"], best["_featnames"],
                                  X_val_df = X_val, y_val=y_val,
                                    results_dir=RESULTS_DIR, save_suffix="best_test")

print("BASELINE:", baseline_params, baseline_metrics)
print("CURATED:", best, test_metrics)




=== BIASED OBSERVED HELOC DATASET ===
Requested target positive fraction: 0.050
Achieved positive fraction:        0.050
Counts:
 target
0    5136
1     270
Name: count, dtype: int64

=== APS COLUMN SETUP ===
Total features: 23
Categorical columns: 0
Binary columns: 0
True numeric columns: 23

=== DATA SPLITS ===
Train: 3,784 samples
  - Minority: 189
  - Majority: 3,595
Val: 811 samples
  - Minority: 41
  - Majority: 770
Test: 811 samples
  - Minority: 40
  - Majority: 771
Cases (minority): 189
Controls (majority): 3,595
→ Building preprocessor w/ imputation:
   • Cat: impute(most_frequent) + OHE on: []
   • Num: impute(median) + scale on: ['ExternalRiskEstimate', 'MSinceOldestTradeOpen', 'MSinceMostRecentTradeOpen', 'AverageMInFile', 'NumSatisfactoryTrades', 'NumTrades60Ever2DerogPubRec', 'NumTrades90Ever2DerogPubRec', 'PercentTradesNeverDelq', 'MSinceMostRecentDelq', 'MaxDelq2PublicRecLast12M', 'MaxDelqEver', 'NumTotalTrades', 'NumTradesOpeninLast12M', 'PercentInstallTrades', 'MSin

Computing distances: 100%|██████████| 4/4 [00:00<00:00, 1242.94it/s]

  Distance matrix shape: (3595, 189)
  Distance range: [1.098, 33.795]
  Distance mean: 6.611

Saving to HDF5: ./precomputed_distances/distances_fico_heloc_obs050_seed_123.h5
  ✓ Saved 2.4 MB

✓ Saved distances to: ./precomputed_distances/distances_fico_heloc_obs050_seed_123.h5
Finetuning OCT (with imputation) for best PR-AUC
→ Building preprocessor w/ imputation:
   • Cat: impute(most_frequent) + OHE on: []
   • Num: impute(median) + scale on: ['ExternalRiskEstimate', 'MSinceOldestTradeOpen', 'MSinceMostRecentTradeOpen', 'AverageMInFile', 'NumSatisfactoryTrades', 'NumTrades60Ever2DerogPubRec', 'NumTrades90Ever2DerogPubRec', 'PercentTradesNeverDelq', 'MSinceMostRecentDelq', 'MaxDelq2PublicRecLast12M', 'MaxDelqEver', 'NumTotalTrades', 'NumTradesOpeninLast12M', 'PercentInstallTrades', 'MSinceMostRecentInqexcl7days', 'NumInqLast6M', 'NumInqLast6Mexcl7days', 'NetFractionRevolvingBurden', 'NetFractionInstallBurden', 'NumRevolvingTradesWBalance', 'NumInstallTradesWBalance', 'NumBank2NatlTrad

Best params: (7, 25, 1e-05) @ val PR-AUC: 0.1658
Test dataset for OCT application: 811 samples
✓ Predictions completed
Computing optimal thresholds on validation set (811 samples)
  Applied validation-set thresholds to test set for evaluation
✓ Saved OCT predictions to: ./heloc_obs005/predictions/oct_predictions_baseline.csv
✓ Saved split table (7 splits) to: ./heloc_obs005/oct_tree_baseline_splits.csv
AUC score: 0.669
PR-AUC (Average Precision): 0.129
Best MCC (test set, threshold from val): 0.204 @ threshold=0.102067
Sensitivity (Recall) @MCC*: 0.225
Specificity @MCC*: 0.966
Balanced (G-mean) recall (test set, threshold from val): 0.600
Balanced (G-mean) specificity (test set, threshold from val): 0.698
Number of leaves: 8

GLOBAL K-CENTER MATCHING CONFIGURATION:
  case_weighting: None
  use_adaptive_pool: True
  seed_method: random
  matching_ratio: 1:1

Global Statistics:
  Cases (minority): 189
  Controls (majority): 3,595
  Ratio: 19.02:1

  Preprocessing:
    Features: 23
    Pr

leaf global d_nn: 100%|██████████| 5/5 [00:00<00:00, 109.27it/s]

[leaf global] saved:
  ./precomputed_distances/global_dnn_fico_heloc_obs050_seed_123/leaf_global_dnn_matrix.npy
  ./precomputed_distances/global_dnn_fico_heloc_obs050_seed_123/leaf_global_dnn_enrolids.npy
    ✓ Control-control distances computed and saved

  K-Center Configuration:
    M (candidate pool size): 1,797 / 3,595 (50.0%)
    Cases to match: 189
    Seed method: random
    Adaptive pool: True
    Case weighting: None

  Running two-stage k-center matching (1:1)...


  Seed selection method: 'random'
    Random seed selected: index 3418
  Auto-computed tau (95th percentile of best distances): 3.4068
  Adaptive pool stopped at 935 candidates (max cost: 2.9831)
    ✓ Matching complete!
    Cases matched: 189
    Total selected controls: 189 (unique: 189)
    Mean matching cost: 2.3763
    Sampling time: 0.315s
    Memory used: 1764.7 MB (Δ +25.3 MB)

BUILDING UNDERSAMPLED TRAINING DATASET

✓ Collected all minority samples: 189
✓ Collected selected majority samples: 189

✓ Undersampled dataset created:
   Total samples: 378
   Minority: 189
   Majority: 189
   Ratio (maj:min): 1.00:1

Class distribution:
target
0    189
1    189
Name: count, dtype: int64
Finetuning OCT (with imputation) for best PR-AUC
→ Building preprocessor w/ imputation:
   • Cat: impute(most_frequent) + OHE on: []
   • Num: impute(median) + scale on: ['ExternalRiskEstimate', 'MSinceOldestTradeOpen', 'MSinceMostRecentTradeOpen', 'AverageMInFile', 'NumSatisfactoryTrades', 'NumTrades

In [ ]:
# 5) Optional imblearn baseline (example: random undersampling)
def fit_eval_imblearn(method, method_name):
    X_tr = train_heloc[feature_cols_heloc].copy()
    y_tr = train_heloc["target"].copy()

    # impute BEFORE resampling (imblearn often can't handle NaN)
    imp = SimpleImputer(strategy="median")
    X_tr_imp = pd.DataFrame(imp.fit_transform(X_tr), columns=feature_cols_heloc)

    X_rs, y_rs = method.fit_resample(X_tr_imp, y_tr)

    # feed into finetune_oct_impute (it will impute again, but harmless)
    X_rs_df = pd.DataFrame(X_rs, columns=feature_cols_heloc)
    model_b, params_b, _, preproc_b, featnames_b = finetune_oct_impute(
        X_train=X_rs_df, y_train=pd.Series(y_rs),
        X_val=X_val, y_val=y_val,
        categorical_cols=CAT_COLUMNS_HELOC,
        numeric_cols=TRUE_NUM_COLUMNS_HELOC,
        depths=OCT_DEPTHS, minbuckets=OCT_MINBUCKETS, cps=OCT_CPS,
    )
    metrics_b = evaluate_binary_oct(model_b, X_test, y_test, preproc_b, featnames_b,
                                   results_dir=RESULTS_DIR, ratio=None)
    return params_b, metrics_b

# params_rus, metrics_rus = fit_eval_imblearn(RandomUnderSampler(sampling_strategy=1.0, random_state=TRAIN_TEST_SEED),
#                                            "RandomUnderSampler(1:1)")
